# Imports

In [ ]:
import json
import requests

import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path

# Params

In [ ]:
PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "raw"

QUERY_PATH = OUTPUT_DIR / "arxiv_query.json"

In [ ]:
with open(
    QUERY_PATH,
    encoding="utf-8",
) as file:
    query = json.load(file)

QUERY = query["query"]

# QUERY = "multi agent systems"

MAX_RESULTS = 10

In [4]:
BASE_URL = "http://export.arxiv.org/api/query"

params = {
    "search_query": f"all:{QUERY}",
    "start": 0,
    "max_results": MAX_RESULTS,
    "sortBy": "submittedDate",
    "sortOrder": "descending",
}

# Call API

In [ ]:
response = requests.get(
    BASE_URL,
    params=params,
    timeout=30,
)

response.raise_for_status()

print(response.status_code)

In [9]:
print(response.text[:1500])

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/nG/nxoRArze+mGJ09XdIDNjWW8w</id>
  <title>arXiv Query: search_query=all:multi OR all:agent OR all:systems&amp;id_list=&amp;start=0&amp;max_results=10</title>
  <updated>2026-07-30T11:47:10Z</updated>
  <link href="https://arxiv.org/api/query?search_query=all:multi+OR+(all:agent+OR+all:systems)&amp;start=0&amp;max_results=10&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>10</opensearch:itemsPerPage>
  <opensearch:totalResults>1031733</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2607.27206v1</id>
    <title>Practical Quantum Topological Data Analysis with Applications to High-Dimensional Feature Extraction and Time Series Analysis</title>
    <updated>2026-07-29T17:59:59Z</updated>
    <l

In [10]:
root = ET.fromstring(response.text)

In [11]:
namespace = {
    "atom": "http://www.w3.org/2005/Atom"
}

In [12]:
entries = root.findall(
    "atom:entry",
    namespace,
)

len(entries)

10

# Build a table from response

In [13]:
papers = []

for entry in entries:

    title = entry.find(
        "atom:title",
        namespace,
    ).text.strip()

    summary = entry.find(
        "atom:summary",
        namespace,
    ).text.strip()

    published = entry.find(
        "atom:published",
        namespace,
    ).text

    url = entry.find(
        "atom:id",
        namespace,
    ).text

    authors = [
        author.find(
            "atom:name",
            namespace,
        ).text
        for author in entry.findall(
            "atom:author",
            namespace,
        )
    ]

    papers.append(
        {
            "title": title,
            "authors": ", ".join(authors),
            "published": published,
            "summary": summary,
            "url": url,
        }
    )

In [14]:
df = pd.DataFrame(papers)

In [15]:
df

,title,authors,published,summary,url
0,Practical Quantum Topological Data Analysis wi...,"Jason Iaconis, Sayonee Ray, Samwel Sekwao, Cla...",2026-07-29T17:59:59Z,Topological data analysis (TDA) provides a pow...,http://arxiv.org/abs/2607.27206v1
1,Mental World Modeling,"Hao Fei, Yiran Zhao",2026-07-29T17:59:39Z,World models enable a predictive substrate for...,http://arxiv.org/abs/2607.27201v1
2,Quantum-Geometric Raman Response in Multiorbit...,"Wai Ting Tai, Martin Claassen",2026-07-29T17:59:32Z,Flat-band materials host rich collective pheno...,http://arxiv.org/abs/2607.27200v1
3,Settling the Optimal Exponent Relating Sumsets...,"Haowei Lin, Shanda Li",2026-07-29T17:59:19Z,For a finite nonempty subset $A$ of an abelian...,http://arxiv.org/abs/2607.27199v1
4,From Classification to Regression: Using a Fru...,"Shady E. Ahmed, Panos Stinis",2026-07-29T17:58:05Z,We present a novel approach to regression task...,http://arxiv.org/abs/2607.27196v1
5,VidMap: Exploiting Temporal Structure for Vide...,"Zador Pataki, Paul-Edouard Sarlin, Marc Pollefeys",2026-07-29T17:58:04Z,Accurately recovering the camera's calibration...,http://arxiv.org/abs/2607.27194v1
6,Non-Minimally Coupled Chain Inflation at High ...,"Miguel Barroso Varela, Orfeu Bertolami, Kather...",2026-07-29T17:58:00Z,Chain inflation offers an alternative to stand...,http://arxiv.org/abs/2607.27193v1
7,The Fidelity and Feedback Traps: The Case for ...,"Nikki L. B. Freeman, Yating Zou, Kyungbok Lee,...",2026-07-29T17:57:57Z,Digital twins for health may be used to compar...,http://arxiv.org/abs/2607.27192v1
8,Can AI agents conduct open-ended AI research? ...,"Peter Kirgis, Sayash Kapoor, Andrew Schwartz, ...",2026-07-29T17:57:19Z,Forecasts of explosive AI progress hinge on AI...,http://arxiv.org/abs/2607.27191v1
9,APEX-Accounting,"Julien Benchek, Austin Bennett, Jasmin Kern, R...",2026-07-29T17:56:49Z,"We introduce APEX-Accounting, a benchmark buil...",http://arxiv.org/abs/2607.27189v1


# Save results

In [16]:
df.to_csv(
    OUTPUT_DIR / "arxiv_search.csv",
    index=False,
)